In [4]:
# Install required libraries
!pip install transformers[torch] datasets accelerate scikit-learn pandas -U

# Import necessary modules
import os
import torch
import numpy as np
import pandas as pd
from datasets import Dataset, DatasetDict
from transformers import (
    AutoConfig,
    AutoModelForSequenceClassification,
    AutoTokenizer,
    TrainingArguments,
    Trainer,
)
from torch.optim import AdamW
from sklearn.metrics import f1_score, accuracy_score

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.2/91.2 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 101.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 130.5 MB/s eta 0:00:00
  Attempting uninstall: scikit-learn
    Found existing installation: scikit-learn 1.6.1
    Uninstalling scikit-learn-1.6.1:
      Successfully uninstalled scikit-learn-1.6.1
  Attempting uninstall: pandas
    Found existing installation: pandas 2.2.2
    Uninstalling pandas-2.2.2:
      Successfully uninstalled pandas-2.2.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.3.2 which is incompatible.
cudf-cu12 25.6.0 requires pandas<2.2.4dev0,>=2.0, but you have pandas 2.3.2 which is incompatible.
dask-cudf-cu12 25.6.0 requires pandas<2.2.4dev0,>=2.0, but you have 

In [2]:
import torch
import torch.nn as nn
from transformers import Trainer
from torch.cuda.amp import autocast

class FGM:
    """
    Fast Gradient Method (FGM) for adversarial training.
    Adds a perturbation to the model's embeddings.
    """
    def __init__(self, model):
        self.model = model
        self.backup = {}

    def attack(self, epsilon=1.0):
        # Access the embeddings layer based on the model's specific architecture
        # For XLMRoberta, the embeddings are usually under model.roberta.embeddings or model.embeddings
        # We will try model.base_model.embeddings first as multilingual-e5-base is likely a variant of roberta
        emb_name = None
        for name, param in self.model.named_parameters():
            if 'embeddings.word_embeddings' in name:
                emb_name = name.split('.weight')[0] # Get the module name
                break

        if emb_name is None:
            # Fallback for models where embeddings might be directly under the model
            for name, param in self.model.named_parameters():
                if 'embeddings' in name and 'word_embeddings' in name:
                     emb_name = name.split('.weight')[0]
                     break

        if emb_name is None:
             raise AttributeError("Could not find word embeddings layer in the model.")


        for name, param in self.model.named_parameters():
            if param.requires_grad and emb_name in name:
                self.backup[name] = param.data.clone()
                norm = torch.norm(param.grad)
                if norm != 0 and not torch.isnan(norm):
                    r_at = epsilon * param.grad / norm
                    param.data.add_(r_at)

    def restore(self):
        # Access the embeddings layer based on the model's specific architecture
        emb_name = None
        for name, param in self.model.named_parameters():
            if 'embeddings.word_embeddings' in name:
                emb_name = name.split('.weight')[0]
                break

        if emb_name is None:
            # Fallback for models where embeddings might be directly under the model
            for name, param in self.model.named_parameters():
                if 'embeddings' in name and 'word_embeddings' in name:
                     emb_name = name.split('.weight')[0]
                     break

        if emb_name is None:
             raise AttributeError("Could not find word embeddings layer in the model.")

        for name, param in self.model.named_parameters():
            if param.requires_grad and emb_name in name:
                assert name in self.backup
                param.data = self.backup[name]
        self.backup = {}


class TACT_FGM_LLRD_Trainer(Trainer):
    """
    Final Custom Trainer that combines:
    1. FGM Adversarial Training
    2. Layer-wise Learning Rate Decay (LLRD)
    """
    def __init__(self, *args, fgm_epsilon=1.0, **kwargs):
        super().__init__(*args, **kwargs)
        # Initialize FGM with the model
        self.fgm = FGM(self.model)
        self.fgm_epsilon = fgm_epsilon

    def training_step(self, model, inputs, num_items_in_batch=None):
        """
        Overrides the training_step to implement FGM.
        Accepts num_items_in_batch for compatibility with Trainer's internal calls.
        """
        model.train()
        inputs = self._prepare_inputs(inputs)

        # --- Standard Forward Pass and Loss Calculation ---
        with autocast(enabled=self.args.fp16):
            outputs = model(**inputs)
            loss = outputs.loss / self.args.gradient_accumulation_steps

        # --- Adversarial Training Step (FGM) ---
        # Calculate gradients for the original loss
        if self.args.fp16:
            self.accelerator.scaler.scale(loss).backward()
        else:
            loss.backward()

        # 1. FGM Attack: Add perturbation to embeddings
        self.fgm.attack(epsilon=self.fgm_epsilon)

        # 2. Compute loss on the adversarial example
        with autocast(enabled=self.args.fp16):
             adv_outputs = model(**inputs)
             adv_loss = adv_outputs.loss / self.args.gradient_accumulation_steps


        # 3. Perform backward pass for adversarial loss
        if self.args.fp16:
            self.accelerator.scaler.scale(adv_loss).backward()
        else:
            adv_loss.backward()


        # 4. Restore the original embeddings
        self.fgm.restore()

        # The optimizer step and scheduler step are handled by the Trainer's main loop
        # We return the total loss for logging (original + adversarial)
        # Note: The gradients are accumulated across both forward passes before the optimizer step.
        return loss.detach() + adv_loss.detach()

    def create_optimizer(self):
        """
        Overrides the create_optimizer method to implement LLRD.
        Adjusted to handle XLMRoberta architecture.
        """
        learning_rate = self.args.learning_rate
        # Access the correct base model attribute
        # Check if the model has a base_model_prefix attribute, otherwise assume 'base_model' or similar
        base_model = None
        # --- START: Lines to potentially change for BanglaBERT (ELECTRA) ---
        # Inspect the model structure to find the correct attribute name for the base model (e.g., 'electra')
        if hasattr(self.model, self.model.base_model_prefix):
             base_model = getattr(self.model, self.model.base_model_prefix)
        elif hasattr(self.model, 'base_model'): # Common for some models (e.g. BERT)
             base_model = getattr(self.model, 'base_model')
        elif hasattr(self.model, 'roberta'): # Specific to Roberta variants
             base_model = getattr(self.model, 'roberta')
        elif hasattr(self.model, 'electra'): # Add check for ELECTRA base model
             base_model = getattr(self.model, 'electra')
        # --- END: Lines to potentially change for BanglaBERT (ELECTRA) ---
        else:
             raise AttributeError("Could not find the base model attribute (e.g., base_model, roberta, electra).")

        num_layers = base_model.config.num_hidden_layers
        lr_decay_rate = 0.9

        optimizer_grouped_parameters = [
            # Access the correct embeddings layer - This might need adjustment if the embedding layer
            # isn't directly accessible under base_model.embeddings in BanglaBERT/ELECTRA.
            # Check the model structure to confirm.
            {"params": base_model.embeddings.parameters(), "weight_decay": self.args.weight_decay, "lr": learning_rate * (lr_decay_rate ** (num_layers + 1))},
        ]
        # Access the correct encoder layers - This might need adjustment if the encoder layers
        # aren't directly accessible under base_model.encoder.layer in BanglaBERT/ELECTRA.
        # Check the model structure to confirm.
        for i, layer in enumerate(base_model.encoder.layer):
            optimizer_grouped_parameters.append(
                {"params": layer.parameters(), "weight_decay": self.args.weight_decay, "lr": learning_rate * (lr_decay_rate ** (num_layers - i))},
            )
        # The classifier layer access remains the same for most models
        # Check if the model has a classifier attribute
        if hasattr(self.model, 'classifier'):
             optimizer_grouped_parameters.append(
                 {"params": self.model.classifier.parameters(), "weight_decay": self.args.weight_decay, "lr": learning_rate},
             )
        elif hasattr(self.model, 'qa_outputs'): # For QA models
             optimizer_grouped_parameters.append(
                 {"params": self.model.qa_outputs.parameters(), "weight_decay": self.args.weight_decay, "lr": learning_rate},
             )
        else:
             print("Warning: Could not find a standard classifier or QA output layer. LLRD may not be applied to the final layer.")


        self.optimizer = AdamW(
            optimizer_grouped_parameters,
            lr=learning_rate,
            eps=self.args.adam_epsilon,
        )
        return self.optimizer

print("Advanced TACT components (FGM, TACT_FGM_LLRD_Trainer) are defined and updated for XLMRoberta architecture.")

Advanced TACT components (FGM, TACT_FGM_LLRD_Trainer) are defined and updated for XLMRoberta architecture.


In [6]:
# --- Configuration ---
# MODEL_NAME = 'intfloat/multilingual-e5-base'
MODEL_NAME = 'csebuetnlp/banglabert'
TRAIN_FILE = '/content/drive/MyDrive/Colab Notebooks/blp25_hatespeech_subtask_1B_train.tsv'
DEV_FILE = '/content/drive/MyDrive/Colab Notebooks/blp25_hatespeech_subtask_1B_dev.tsv'
TEST_FILE = '/content/drive/MyDrive/Colab Notebooks/blp25_hatespeech_subtask_1B_test.tsv'
OUTPUT_FILE = "subtask_1B.tsv"
# --- Initialize Tokenizer ---
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# --- Helper Functions ---
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    f1 = f1_score(labels, predictions, average='macro')
    acc = accuracy_score(labels, predictions)
    return {'f1': f1, 'accuracy': acc}

def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=128)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/119 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/586 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

TACT Phase 1: Task-Adaptive Fine-Tuning

In [7]:
print("--- TACT Phase 1: Preparing Data for Task-Adaptive Tuning ---")

train_df = pd.read_csv(TRAIN_FILE, sep='\t', keep_default_na=False)
dev_df = pd.read_csv(DEV_FILE, sep='\t', keep_default_na=False)

train_df['binary_label'] = train_df['label'].apply(lambda x: 0 if x == 'None' else 1)
dev_df['binary_label'] = dev_df['label'].apply(lambda x: 0 if x == 'None' else 1)

binary_i2l = {0: 'Not-Hate', 1: 'Hate'}
binary_l2i = {v: k for k, v in binary_i2l.items()}

train_ds_binary = Dataset.from_pandas(train_df[['text', 'binary_label']].rename(columns={'binary_label': 'label'}))
dev_ds_binary = Dataset.from_pandas(dev_df[['text', 'binary_label']].rename(columns={'binary_label': 'label'}))

ds_binary = DatasetDict({'train': train_ds_binary, 'validation': dev_ds_binary})
tokenized_ds_binary = ds_binary.map(tokenize_function, batched=True)

print("\nBinary data prepared successfully.")
print(train_df['binary_label'].value_counts())

--- TACT Phase 1: Preparing Data for Task-Adaptive Tuning ---


Map:   0%|          | 0/35522 [00:00<?, ? examples/s]

Map:   0%|          | 0/2512 [00:00<?, ? examples/s]


Binary data prepared successfully.
binary_label
0    21190
1    14332
Name: count, dtype: int64


Step 5: Train the Task-Adaptive Model
Here, we now instantiate our new, powerful TACT_FGM_LLRD_Trainer.

In [8]:
print("--- TACT Phase 1: Training the Task-Adaptive (Binary) Model ---")

config_binary = AutoConfig.from_pretrained(MODEL_NAME, num_labels=2, id2label=binary_i2l, label2id=binary_l2i)
model_binary = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, config=config_binary)

training_args_binary = TrainingArguments(
    output_dir="./banglabert-binary-fgm",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3, # Reduced to half an epoch
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
)

# Use the new, combined trainer
trainer_binary = TACT_FGM_LLRD_Trainer(
    model=model_binary,
    args=training_args_binary,
    train_dataset=tokenized_ds_binary["train"],
    eval_dataset=tokenized_ds_binary["validation"],
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
)

trainer_binary.train()
print("\n Task-Adaptive Model training complete.")

--- TACT Phase 1: Training the Task-Adaptive (Binary) Model ---


pytorch_model.bin:   0%|          | 0.00/443M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/443M [00:00<?, ?B/s]

Some weights of ElectraForSequenceClassification were not initialized from the model checkpoint at csebuetnlp/banglabert and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/tmp/ipython-input-2701774883.py:76: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `TACT_FGM_LLRD_Trainer.__init__`. Use `processing_class` instead.
  super().__init__(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)


<IPython.core.display.Javascript object>

...

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: shifat-islam-hs (shifat-islam-hs-bangladesh-university-of-engineering-and) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


/tmp/ipython-input-2701774883.py:90: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=self.args.fp16):
/tmp/ipython-input-2701774883.py:105: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=self.args.fp16):


Epoch,Training Loss,Validation Loss,F1,Accuracy
1,1.033900,0.459883,0.773053,0.785828
2,0.963500,0.443697,0.783726,0.792994
3,0.894300,0.439996,0.780278,0.791003


/tmp/ipython-input-2701774883.py:90: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=self.args.fp16):
/tmp/ipython-input-2701774883.py:105: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=self.args.fp16):
/tmp/ipython-input-2701774883.py:90: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=self.args.fp16):
/tmp/ipython-input-2701774883.py:105: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=self.args.fp16):



 Task-Adaptive Model training complete.


TACT Phase 2: Category-Tuned Fine-Tuning
Step 6: Prepare Data for Category-Tuned Tuning

In [4]:
print("--- TACT Phase 2: Preparing Data for Category-Tuned Tuning ---")

train_df_multi = train_df[train_df['label'] != 'None'].copy()
dev_df_multi = dev_df[dev_df['label'] != 'None'].copy()

multi_l2i = {'Individual': 0, 'Community': 1, 'Organization': 2, 'Society': 3}
multi_i2l = {v: k for k, v in multi_l2i.items()}

train_df_multi['label'] = train_df_multi['label'].map(multi_l2i)
dev_df_multi['label'] = dev_df_multi['label'].map(multi_l2i)

train_ds_multi = Dataset.from_pandas(train_df_multi[['text', 'label']])
dev_ds_multi = Dataset.from_pandas(dev_df_multi[['text', 'label']])

ds_multi = DatasetDict({'train': train_ds_multi, 'validation': dev_ds_multi})
tokenized_ds_multi = ds_multi.map(tokenize_function, batched=True)

print("\nMulti-class data prepared successfully.")
print(train_df_multi['label'].map(multi_i2l).value_counts())

--- TACT Phase 2: Preparing Data for Category-Tuned Tuning ---


NameError: name 'train_df' is not defined

Step 7: Train the Category-Tuned Model
Again, we use our new TACT_FGM_LLRD_Trainer for the second phase of training.

In [3]:
print("--- TACT Phase 2: Training the Category-Tuned (Multi-Class) Model ---")

config_multi = AutoConfig.from_pretrained(MODEL_NAME, num_labels=4, id2label=multi_i2l, label2id=multi_l2i)
model_multi = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, config=config_multi)

training_args_multi = TrainingArguments(
    output_dir="./banglabert-multi-fgm",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=4,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
)

# Use the new, combined trainer
trainer_multi = TACT_FGM_LLRD_Trainer(
    model=model_multi,
    args=training_args_multi,
    train_dataset=tokenized_ds_multi["train"],
    eval_dataset=tokenized_ds_multi["validation"],
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
)

trainer_multi.train()
print("\n Category-Tuned Model training complete.")

--- TACT Phase 2: Training the Category-Tuned (Multi-Class) Model ---


NameError: name 'AutoConfig' is not defined

Final Step: Prediction using the TACT Pipeline
Step 8: Predict on Test Set and Save Results
The prediction step remains the same, as it only uses the final trained models and is not involved in the training loop.

In [ ]:
print("--- Final Step: Predicting on the Test Set with the TACT Pipeline ---")

test_df = pd.read_csv(TEST_FILE, sep='\t', keep_default_na=False)
test_dataset = Dataset.from_pandas(test_df)
tokenized_test_dataset = test_dataset.map(tokenize_function, batched=True)

print("Predicting Hate vs. Not-Hate...")
binary_predictions = trainer_binary.predict(tokenized_test_dataset)
binary_preds = np.argmax(binary_predictions.predictions, axis=1)

hate_indices = [i for i, pred in enumerate(binary_preds) if pred == 1]
hate_texts_dataset = tokenized_test_dataset.select(hate_indices)
print(f"Found {len(hate_indices)} potential hate speech instances to classify further.")

if len(hate_indices) > 0:
    multi_predictions = trainer_multi.predict(hate_texts_dataset)
    multi_preds_raw = np.argmax(multi_predictions.predictions, axis=1)
    multi_preds = [multi_i2l[p] for p in multi_preds_raw]
else:
    multi_preds = []

final_labels = []
multi_pred_idx = 0
for i in range(len(test_df)):
    if binary_preds[i] == 0:
        final_labels.append('None')
    else:
        final_labels.append(multi_preds[multi_pred_idx])
        multi_pred_idx += 1

output_df = pd.DataFrame({
    'id': test_df['id'],
    'label': final_labels,
    'model_name': f"{MODEL_NAME}-TACT-FGM"
})
output_df.to_csv(OUTPUT_FILE, sep='\t', index=False)

print(f"\n Predictions successfully saved to {OUTPUT_FILE}")
print("\n--- Final Predictions Head ---")
print(output_df.head())
print("\n--- Final Label Distribution ---")
print(output_df['label'].value_counts())

In [ ]:
print("Evaluating the best model on the validation set...")
eval_results = trainer_multi.evaluate(tokenized_ds_multi["validation"])

print("\n--- Final Evaluation Results ---")
for key, value in eval_results.items():
    print(f"{key}: {value:.4f}")

In [ ]:
import pandas as pd
import numpy as np
from datasets import Dataset

# --- Step 8: Predicting on the Test Set and Saving Final Logits ---

print("--- Predicting on the Test Set and Saving Final Logits ---")

# --- Define the final 5-class structure for our output file ---
# # This ensures a consistent format for the logit columns.
# FINAL_LABEL_MAP = {
#     'Society': 0,
#     'Organization': 1,
#     'None': 2,
#     'Individual': 3,
#     'Community': 4
# }

# --- Load and prepare the test data ---
test_df = pd.read_csv(TEST_FILE, sep='\t', keep_default_na=False)
test_dataset = Dataset.from_pandas(test_df)
tokenized_test_dataset = test_dataset.map(tokenize_function, batched=True)

# --- Stage 1: Get raw logit predictions from the binary model ---
print("Getting logits from Stage 1 (Binary) model...")
# Remove the 'label' column before predicting
tokenized_test_dataset_for_binary_prediction = tokenized_test_dataset.remove_columns(["label"])
binary_predictions = trainer_binary.predict(tokenized_test_dataset_for_binary_prediction)
binary_logits = binary_predictions.predictions  # Shape: (num_samples, 2)
binary_preds = np.argmax(binary_logits, axis=1) # Get the final decision (0 or 1)

# --- Stage 2: Get raw logit predictions from the multi-class model ---
print("Getting logits from Stage 2 (Multi-Class) model...")
hate_indices = [i for i, pred in enumerate(binary_preds) if pred == 1]
hate_texts_dataset = tokenized_test_dataset.select(hate_indices)

if len(hate_indices) > 0:
    # Remove the 'label' column before predicting
    hate_texts_dataset_for_multi_prediction = hate_texts_dataset.remove_columns(["label"])
    multi_predictions = trainer_multi.predict(hate_texts_dataset_for_multi_prediction)
    multi_logits = multi_predictions.predictions # Shape: (num_hate_samples, 4)
else:
    multi_logits = []

# --- Stage 3: Construct the final 5-dimensional logit array ---
print("Constructing final 5-dimensional logits...")
final_logits = []
multi_logit_idx = 0

# Mapping from the multi-class model's output index to the final 5-class index
# multi_i2l = {0: 'Individual', 1: 'Community', 2: 'Organization', 3: 'Society'}
# FINAL_LABEL_MAP = {'Society': 0, 'Organization': 1, 'None': 2, 'Individual': 3, 'Community': 4}
# multi_to_final_idx_map = {
#     3: FINAL_LABEL_MAP['Society'],      # Society logit
#     2: FINAL_LABEL_MAP['Organization'], # Organization logit
#     0: FINAL_LABEL_MAP['Individual'],   # Individual logit
#     1: FINAL_LABEL_MAP['Community']     # Community logit
# }


for i in range(len(test_df)):
    # Create a placeholder array for the 5 final logits, filled with a very low number
    # This ensures that unassigned classes have a very low probability.
    constructed_logit = np.full(5, -100.0)

    if binary_preds[i] == 0:  # Predicted as 'Not-Hate' by the binary model
        # The logit for 'None' in the final 5-class output comes from the binary model's 'Not-Hate' logit.
        # The logits for the other 4 hate categories are very low.
        constructed_logit[0] = binary_logits[i][0] # Assuming index 0 of binary_logits is Not-Hate
    else:  # Predicted as 'Hate' by the binary model
        # The logit for 'None' is the binary model's 'Not-Hate' logit (should be low)
        #constructed_logit[0] = binary_logits[i][0] # Assuming index 0 of binary_logits is Not-Hate

        # Get the corresponding 4-class logits for this sample from the multi-class model
        current_multi_logits = multi_logits[multi_logit_idx]

        # Place the 4 hate-category logits into their correct positions in the 5-dimensional array.
        # Based on the multi_i2l mapping {0: 'Individual', 1: 'Community', 2: 'Organization', 3: 'Society'}
        # and the desired final order {Society, Organization, None, Individual, Community}
        # We need to map the multi-class logits to the correct indices in the 5-dimensional array.
        # Assuming the desired final order is: 0: Society, 1: Organization, 2: None, 3: Individual, 4: Community
        # multi_logits[0] (Individual) goes to index 3
        # multi_logits[1] (Community) goes to index 4
        # multi_logits[2] (Organization) goes to index 1
        # multi_logits[3] (Society) goes to index 0

        constructed_logit[3] = current_multi_logits[0]  # Individual
        constructed_logit[4] = current_multi_logits[1]  # Community
        constructed_logit[1] = current_multi_logits[2]  # Organization
        constructed_logit[0] = current_multi_logits[3]  # Society

        # The logit for 'None' is the binary model's 'Not-Hate' logit (should be low)
        constructed_logit[2] = binary_logits[i][0] # Assuming index 0 of binary_logits is Not-Hate


        multi_logit_idx += 1

    final_logits.append(constructed_logit)

# --- Step 4: Save the final logits to a .tsv file ---
print("Saving final logits to file...")
output_predict_file = os.path.join("./", "subtask_1B_logits_pred advanced (latest).tsv")

# Convert the list of numpy arrays to a list of lists for easier writing
logits_list_to_write = [list(row) for row in final_logits]
ids = test_df['id'].tolist()

with open(output_predict_file, "w") as writer:
    writer.write("id\tlogits\tmodel_name\n")
    for index, logits in enumerate(logits_list_to_write):
        # Convert each logit value to a string and join with spaces
        logits_str = ' '.join(map(str, logits))
        writer.write(f"{ids[index]}\t{logits_str}\t{MODEL_NAME}\n")

print(f"\n Logit predictions successfully saved to {output_predict_file}")

# Optional: Display a sample of what was saved
print("\n--- Sample of Saved Data ---")
sample_df = pd.read_csv(output_predict_file, sep='\t')
print(sample_df.head())